# TMDB Movie Data Pipeline and EDA

This notebook fetches at least 20 movies from the TMDB Discover Movies endpoint,
stores them in a SQLite database, then performs Exploratory Data Analysis (EDA).

In [ ]:
# Install required libraries if needed
# !pip install requests pandas

import requests
import pandas as pd
import sqlite3
import json

## Task 1 - Build the Pipeline

In [ ]:
# Replace the value below with your own TMDB API key
# Get your free API key at: https://www.themoviedb.org/settings/api
API_KEY = 'YOUR_TMDB_API_KEY_HERE'

In [ ]:
# Fetch movies from TMDB Discover Movies endpoint
base_url = 'https://api.themoviedb.org/3/discover/movie'
all_movies = []

# Fetch 2 pages (20 movies per page = 40 movies total)
for page in range(1, 3):
    params = {
        'api_key': API_KEY,
        'language': 'en-US',
        'sort_by': 'popularity.desc',
        'page': page
    }
    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()
    all_movies.extend(data['results'])

print(f'Total movies fetched: {len(all_movies)}')

In [ ]:
# Create a DataFrame from the fetched movies
df = pd.DataFrame(all_movies)

# Select relevant columns
cols = ['id', 'title', 'release_date', 'popularity', 'vote_average',
        'vote_count', 'genre_ids', 'overview', 'original_language']
df = df[[c for c in cols if c in df.columns]]

# Convert genre_ids list to string for SQLite storage
df['genre_ids'] = df['genre_ids'].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)

print(df.shape)
print(df.dtypes)

In [ ]:
# Save DataFrame to SQLite database table called 'movies'
conn = sqlite3.connect('movies.db')
df.to_sql('movies', conn, if_exists='replace', index=False)
conn.commit()
conn.close()
print('Data saved to SQLite database successfully.')

## Task 2 - Perform EDA

In [ ]:
# Load the movies table back into a DataFrame
conn = sqlite3.connect('movies.db')
movies_df = pd.read_sql('SELECT * FROM movies', conn)
conn.close()
print('Data loaded from SQLite successfully.')

In [ ]:
# Display the first 5 rows
movies_df.head()

In [ ]:
# Show summary statistics for numeric columns
movies_df.describe()

In [ ]:
# Count the number of movies per genre
# genre_ids is stored as JSON string, expand and count
import ast

genre_series = movies_df['genre_ids'].apply(lambda x: json.loads(x) if isinstance(x, str) else [])
genre_counts = {}
for genres in genre_series:
    for gid in genres:
        genre_counts[gid] = genre_counts.get(gid, 0) + 1

genre_df = pd.DataFrame(list(genre_counts.items()), columns=['genre_id', 'count'])
genre_df = genre_df.sort_values('count', ascending=False).reset_index(drop=True)
print('Movies per genre (by genre_id):')
print(genre_df)

In [ ]:
# Identify columns with missing values
missing = movies_df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print('No missing values found in any column.')
else:
    print('Columns with missing values:')
    print(missing)